In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

with open("processed-scheduled-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    scheduled_data = list(csv_reader)

len(actual_data), len(scheduled_data)

(74063, 12176653)

In [2]:
# convert string to datetime objects
from datetime import datetime, timedelta

from tqdm import tqdm

keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]
for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    for key in keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

keys = ["Scheduled In Date/Time", "Scheduled Out Date/Time"]
for row in tqdm(scheduled_data):
    row["PTID"] = int(row["PTID"])
    for key in keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

100%|██████████| 12176653/12176653 [01:17<00:00, 156644.64it/s]


In [3]:
actual_data[0], scheduled_data[0]

({'PTID': 26053,
  'Name': 'MOUNTAIN-SWANROAD_115_104-3',
  'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
  'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
  'Voltage': '115',
  'FirstBus': 'MOUNTAIN',
  'SecondBus': 'SWANROAD'},
 {'PTID': 25190,
  'Name': 'FARRAGUT-E13THSTA_345_45',
  'Scheduled In Date/Time': datetime.datetime(2005, 2, 1, 0, 59, 59),
  'Scheduled Out Date/Time': datetime.datetime(2005, 1, 31, 22, 12, 32)})

In [4]:
# sort both datasets
actual_data.sort(key=lambda row: row["OutDatetime"])
scheduled_data.sort(key=lambda row: row["Scheduled Out Date/Time"])

In [5]:
# setting type algorithm
time_delta = timedelta(hours=1)
start_index = 0
end_index = 0
for row in tqdm(actual_data):
    outage_date = row["OutDatetime"]
    # update start index
    while (
        scheduled_data[start_index]["Scheduled Out Date/Time"]
        > outage_date - time_delta
    ):
        start_index += 1
    # update end index
    while (
        scheduled_data[end_index]["Scheduled Out Date/Time"] < outage_date - time_delta
    ):
        end_index += 1
    # check if PTID exists in that interval
    row["OutageType"] = "Auto"
    for scheduled_row in scheduled_data[start_index:end_index]:
        if scheduled_row["PTID"] == row["PTID"]:
            row["OutageType"] = "Planned"
            break

100%|██████████| 74063/74063 [5:23:16<00:00,  3.82it/s]  


In [6]:
len([row for row in actual_data if row.get("OutageType") == "Planned"])

65376

In [7]:
len([row for row in actual_data if row.get("OutageType") == "Auto"])

8687

In [8]:
from pathlib import Path

import pandas as pd

actual_outage_csv_path = Path("processed-actual-outages.csv")

rows = sorted(actual_data, key=lambda x: x["OutDatetime"])
df = pd.DataFrame(rows)

# write out; use the existing json path with a .csv suffix
df.to_csv(actual_outage_csv_path, index=False)

print(f"saved {len(df)} rows to {actual_outage_csv_path}")

saved 74063 rows to processed-actual-outages.csv
